# 11 - Final model: SVM + CatBoost on iterative-imputed features

This is our best model. It combines the two discoveries that actually moved the score:

1. **Iterative imputation** of the 50%-missing `eog_burst_index` column (the single
   biggest lever - took the SVM from 0.832 to 0.842 CV, and 0.839 -> 0.855 on Kaggle).
2. A **soft-voting ensemble of SVM + CatBoost**, both trained on those imputed features.
   Ensembling only started helping *after* imputation made CatBoost strong enough.

### The story in numbers (all repeated-CV macro-F1, the competition metric)
| Step | What changed | CV macro-F1 |
|---|---|---|
| baseline | tuned SVM, median-imputed | 0.832 |
| **imputation** | tuned SVM, *iterative*-imputed | **0.842** |
| CatBoost (native NaN) | - | 0.822 |
| CatBoost (iterative-imputed) | imputation helps trees too | 0.839 |
| **ensemble** | SVM + CatBoost, iterative-imputed | **~0.845** |

### Why this works (and why the earlier ensemble didn't)
A soft-voting ensemble helps only when members are **both strong AND make different
mistakes**. Earlier, CatBoost (0.822) was too weak and the boosted trees were
correlated with each other, so blending added nothing beyond CV noise. Iterative
imputation lifted CatBoost to 0.839 - now SVM (kernel model) and CatBoost (boosted
trees) are two *strong, different-family* models, so averaging their probabilities
cancels uncorrelated errors. We verified the gain is real: it won **15/15 folds**
across 3 seeds, paired t-test **p ~ 3e-5** (vs the old ensemble whose gain sat inside
the noise band and lost on the leaderboard).

## Step 0 - Imports and data
`enable_iterative_imputer` must be imported before `IterativeImputer` (it's still an
experimental sklearn feature).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer   # noqa: F401  (required to unlock IterativeImputer)
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report

TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
FEATURES = [c for c in train.columns if c not in ('id', 'sleep_stage')]
y = train['sleep_stage'].to_numpy()
print('train', train.shape, '| test', test.shape)
print('eog_burst_index missing: train %.0f%%  test %.0f%%'
      % (100*train.eog_burst_index.isna().mean(), 100*test.eog_burst_index.isna().mean()))

train (9000, 23) | test (5000, 22)
eog_burst_index missing: train 50%  test 50%


## Step 1 - The imputation lever
The only column with missing values is `eog_burst_index` (~50% missing, one of the
more discriminative features). How we fill it matters more than any model choice:

- **Median imputation** writes one constant into every missing cell - it discards the
  relationship between this column and the other 20.
- **Iterative imputation** treats the column as a regression target and *predicts* each
  missing value from the other features (here with a linear `BayesianRidge` model, run
  inside the pipeline so it is re-fit on each training fold - no leakage).

We tested nonlinear imputers (ExtraTrees, HistGBR, KNN) too; they **overfit** the
imputation and scored worse (0.833-0.835), so the simple linear regressor wins.

In [2]:
def make_imputer():
    # only one column is missing, so this is effectively 'predict eog_burst_index
    # from the other 20 features'. BayesianRidge (linear) generalized best.
    return IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)

## Step 2 - The two base models
Both consume the SAME iterative-imputed features, but they are very different learners,
which is exactly what makes their errors decorrelate:

- **SVM (RBF kernel)** - a kernel/margin model. Needs scaling (distance-based). We set
  `probability=True` so it can output class probabilities for soft voting.
- **CatBoost** - gradient-boosted decision trees. Scale-invariant; imputation gives it
  the recovered `eog_burst_index` signal it couldn't exploit via native NaN handling.

Hyperparameters are the tuned values from notebooks 03 and 06.

In [3]:
def make_svm():
    # impute -> scale -> RBF-SVM (probability=True enables predict_proba for voting)
    return make_pipeline(
        make_imputer(), StandardScaler(),
        SVC(kernel='rbf', C=12, gamma=0.012, probability=True, random_state=42),
    )

def make_catboost():
    # impute -> CatBoost (trees need no scaling)
    return make_pipeline(
        make_imputer(),
        CatBoostClassifier(loss_function='MultiClass', iterations=600, depth=7,
                           learning_rate=0.04, l2_leaf_reg=3,
                           random_state=42, verbose=0, thread_count=-1),
    )

## Step 3 - Validate the ensemble robustly (OPTIONAL, SLOW ~10-15 min)
This cell proves the ensemble beats the SVM alone and that the gain is **not** CV noise.
Method: for several fold seeds we compute **out-of-fold** probabilities (each row
predicted by models that didn't train on it), score the SVM and the SVM+CatBoost
average **per fold**, and run a **paired** t-test (paired because both see identical
folds). Expect ~15/15 fold wins and a tiny p-value.

**You can skip straight to Step 4** to just produce the submission. To run a quick
check instead of the full one, set `SEEDS = [42]` (one seed, ~3-4 min).

In [4]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from scipy import stats

SEEDS = [42, 2024, 7]          # set to [42] for a faster sanity check
svm_fold, ens_fold = [], []
for seed in SEEDS:
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    oof_svm = cross_val_predict(make_svm(),      train[FEATURES], y, cv=cv, method='predict_proba', n_jobs=5)
    oof_cat = cross_val_predict(make_catboost(), train[FEATURES], y, cv=cv, method='predict_proba', n_jobs=5)
    for _, idx in cv.split(train[FEATURES], y):
        svm_fold.append(f1_score(y[idx], oof_svm[idx].argmax(1), average='macro'))
        ens_fold.append(f1_score(y[idx], ((oof_svm[idx]+oof_cat[idx])/2).argmax(1), average='macro'))
    print('seed %-4d  SVM=%.4f  SVM+Cat=%.4f'
          % (seed, f1_score(y, oof_svm.argmax(1), average='macro'),
                   f1_score(y, ((oof_svm+oof_cat)/2).argmax(1), average='macro')), flush=True)

svm_fold, ens_fold = np.array(svm_fold), np.array(ens_fold)
gain = ens_fold - svm_fold
print('\nmean gain (ensemble - SVM) = %+.4f +/- %.4f over %d folds'
      % (gain.mean(), gain.std(), len(gain)))
print('ensemble wins %d/%d folds   paired t-test p = %.2e'
      % ((gain > 0).sum(), len(gain), stats.ttest_rel(ens_fold, svm_fold).pvalue))

seed 42    SVM=0.8402  SVM+Cat=0.8438
seed 2024  SVM=0.8404  SVM+Cat=0.8451
seed 7     SVM=0.8437  SVM+Cat=0.8466

mean gain (ensemble - SVM) = +0.0037 +/- 0.0023 over 15 folds
ensemble wins 15/15 folds   paired t-test p = 3.28e-05


## Step 4 - Build the submission
Refit BOTH models on ALL training rows, average their predicted class probabilities on
the test set (equal weights - searching weights overfit the OOF), take the argmax, and
save. This cell is self-contained: it does not depend on Step 3.

In [5]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

svm = make_svm();      svm.fit(train[FEATURES], y)
cat = make_catboost(); cat.fit(train[FEATURES], y)

proba = (svm.predict_proba(test[FEATURES]) + cat.predict_proba(test[FEATURES])) / 2
pred  = proba.argmax(1).astype(int)

submission = pd.DataFrame({'id': test['id'], 'sleep_stage': pred})
path = os.path.join(OUT_DIR, 'ensemble_svm_catboost_iterimpute_submission.csv')
submission.to_csv(path, index=False)
print('wrote', path, submission.shape)
print('predicted class distribution:', dict(submission.sleep_stage.value_counts().sort_index()))
submission.head()

wrote ../outputs/ensemble_svm_catboost_iterimpute_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1132), 1: np.int64(1280), 2: np.int64(1292), 3: np.int64(1296)}


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,3
4,9004,3


## Takeaways
- **Iterative imputation was the real lever** - it helped every model and made the
  ensemble worthwhile. Fix the data problem before tuning models.
- **SVM + CatBoost (equal-weight soft vote)** is the final model: two strong,
  different-family learners whose errors decorrelate. Verified gain (15/15 folds, p~3e-5).
- **Not worth more time** (all tested, none beat the above by more than noise):
  feature engineering / polynomial / PCA on the SVM, nonlinear imputers, bagging,
  weighted-vote and stacking, extra SVM hyperparameter search.
- **If you want to keep going:** carefully test per-class threshold tuning for the weak
  class 2 (validate with repeated CV before trusting it), or add a third strong,
  decorrelated member - weak members will only drag the vote down.